# **Collect multiple web pages for analysis**

## *Mariam Cook*

## *University of Exeter Centre for Computational Social Science - C2S2*

---


In [ ]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
mylinks = ["https://www.theyworkforyou.com/whall/?id=2023-06-06b.309.0", # 1 Net Zero: 2050 Target
                    "https://www.theyworkforyou.com/debates/?id=2023-06-14b.312.3", # 2 cost of living committee
                    "https://www.theyworkforyou.com/debates/?id=2023-06-19b.569.0", # 3 stop and search
                    "https://www.theyworkforyou.com/whall/?id=2023-06-21a.384.0", # 4 ultra-processed food
                    "https://www.theyworkforyou.com/lords/?id=2023-06-19a.5.2", # 5 Primary care - inequality
                    "https://www.theyworkforyou.com/lords/?id=2023-06-15b.2101.2", # 6 Ukraine: Ministry of Defence Strategy - Question
                    "https://www.theyworkforyou.com/debates/?id=2023-06-15a.474.1", # 7 Pride Month
                    "https://www.theyworkforyou.com/debates/?id=2023-06-21b.801.3", # 8 Schedule - Minimum service levels for certain strikes
                    "https://www.theyworkforyou.com/whall/?id=2023-06-21a.352.0", # 9 Tackling Loneliness and Connecting Communities — [Dr Rupa Huq in the Chair]
                    "https://www.theyworkforyou.com/debates/?id=2023-06-22a.940.0", # 10 Business of the House – in the House of Commons at 10:30 am on 22 June 2023.
                    "https://www.theyworkforyou.com/debates/?id=2023-06-21b.849.1", # 11 Animal Welfare (Kept Animals) – in the House of Commons at 3:13 pm on 21 June 2023.
                    "https://www.theyworkforyou.com/debates/?id=2023-06-21b.786.6", # 12 Engagements Prime Minister – in the House of Commons on 21 June 2023.
                    "https://www.theyworkforyou.com/whall/?id=2023-06-19a.227.0", # 13 Cost of Living: Parental Leave and Pay — [Steve McCabe in the Chair]
                    "https://www.theyworkforyou.com/debates/?id=2023-06-20c.701.0", # 14 Cost of Living support
                    "https://www.theyworkforyou.com/debates/?id=2023-06-15a.514.0", # 15 Migration – in the House of Commons at 3:17 pm on 15 June 2023.
                    "https://www.theyworkforyou.com/debates/?id=2023-06-15a.445.0", # 16 Business of the House – in the House of Commons at 11:02 am on 15 June 2023
                    "https://www.theyworkforyou.com/pbc/2022-23/Energy_Bill/14-0_2023-06-22a.381.2", # 17 Clause 270 - Prohibition of new coal mines
                    "https://www.theyworkforyou.com/debates/?id=2023-06-14b.365.0", # 18 Global Military Operations 14 June 2023
                    "https://www.theyworkforyou.com/debates/?id=2023-06-14b.280.4", # 19 Regional Innovation
                    "https://www.theyworkforyou.com/whall/?id=2023-06-13a.100.0" # Tackling Rogue Builders in Westminster Hall at 4:00 pm on 13 June 2023.
                    ]

In [ ]:
def getwebpages(links):
  multiple_soups = []
  for alink in links:
    webpage = requests.get(alink)
    soup = BeautifulSoup(webpage.content, "html.parser")
    multiple_soups.append(soup)
  return multiple_soups

In [ ]:
multiple_page_soup = getwebpages(mylinks)

In [ ]:
len(multiple_page_soup)

In [ ]:
multiple_page_soup[0] # the soup object contains all the html content

In [ ]:
print(multiple_page_soup[0].text) #the text attribute contains the text

In [ ]:
type(multiple_page_soup[0].text)

## Unfortunately not all debate blocks have debate text, so only get blocks with a speaker name

In [ ]:
debate_blocks = []
speaker_names = []
speaker_positions = []
debate_text = []
for some_soup in multiple_page_soup:
  for hit in some_soup.findAll(attrs={'class' : 'debate-speech'}): #get the whole block of each speech, however it has link text we don't want
      #print(hit.text)
      speaker_name = hit.findAll(attrs={'class' : 'debate-speech__speaker__name'})
      #print(len(speaker_name))
      if(len(speaker_name) == 0):
        print("empty speaker")
        print(hit)
        print(hit.text)
      else:
        print('this has a speaker')
        for item in speaker_name:
          #print(item)
          #print(item.text)
          speaker_names.append(item.text)
          debate_blocks.append(hit.text)
          speaker_position = hit.findAll(attrs={'class' : 'debate-speech__speaker__position'})
          for position_item in speaker_position:
            speaker_positions.append(position_item.text)
          debate_text_items = hit.findAll(attrs={'class' : 'debate-speech__content'})
          for debate_text_item in debate_text_items:
            debate_text.append(debate_text_item.text)

In [ ]:
len(debate_blocks)

In [ ]:
len(speaker_names)

In [ ]:
len(speaker_positions)

In [ ]:
len(debate_text)

In [ ]:
# zip the lists directly into a dataframe
df = pd.DataFrame(list(zip(speaker_names, speaker_positions, debate_text)),
               columns =['Name', 'Position', 'Debate Text'])
df

In [ ]:
party = []
for position in speaker_positions:
  print(position)
  if (('Labour' in position) or ('Shadow' in position)):
    print('Labour')
    #print(position)
    party.append('Labour')
  elif (('Conservative' in position) or ('Minister' in position) or ('Secretary of State' in position) or ('Under-Secretary' in position)):
    #print(position)
    party.append('Conservative')
  elif 'Green' in position:
    print('Green')
    print(position)
    party.append('Green')
  elif 'Liberal Democrat' in position:
    #print('Liberal Democrat')
    #print(position)
    party.append('Liberal Democrat')
  elif (('SNP' in position) or ('Scottish National Party' in position)):
    #print('SNP')
    #print(position)
    party.append('SNP')
  elif 'DUP' in position:
    #print('DUP')
    #print(position)
    party.append('DUP')
  else:
    print('other!!!!!!!!!!!!')
    print(position)
    party.append('other')

In [ ]:
party

In [ ]:
# zip the list directly into a dataframe
df = pd.DataFrame(list(zip(speaker_names, party, speaker_positions, debate_text)),
               columns =['Name', 'Party', 'Position', 'Text'])
df

## Let's see how many 'other' party there is

In [ ]:
df['Party'].value_counts()['other']

In [ ]:
set(df.loc[df['Party'] == 'other']['Position'].values)

### (We can create a script later to deal with the 'other' issue - figuring out what each party is from the above)

# Let's see how many of each party

In [ ]:
df['Party'].value_counts()

# Let's just get the Conservative and Labour MPs

In [ ]:
con_lab = df[df["Party"].isin(['Conservative', 'Labour'])].reset_index(drop=True)
con_lab

# Let's export the Conservative and Labour MPs debate text

In [ ]:
#download the file on the left from the Colab folder to save it
con_lab.to_csv('con_lab.csv')

# Analysis: all parties

In [ ]:
#restriction = 0
#regulation = 1
#pollution = 0
#renewable = 7
#transport = 2
#energy = many
#emissions = 3

energy =[]
renewable =[]

import re
for i,row in df.iterrows():
  #print(row['Debate Text'])
  matches = re.findall('renewable', row['Text'].lower())
  if len(matches)>0:
    print(matches)
  energy.append(len(matches))
  matches2 = re.findall('energy', row['Text'].lower())
  if len(matches2)>0:
    print(matches2)
  renewable.append(len(matches2))

In [ ]:
energy

In [ ]:
len(energy)

In [ ]:
renewable

In [ ]:
# zip the list with results directly into a dataframe
df2 = pd.DataFrame(list(zip(speaker_names, party, speaker_positions, debate_text, energy, renewable)),
               columns =['Name', 'Party', 'Position', 'Debate Text', 'Mention Energy', 'Mention Renewable'])
df2

In [ ]:
# Import the necessary command first
#from scipy.stats import pearsonr # Pearson's r

In [ ]:
# Let's get the mean number of mentions per party
df2.groupby('Party')['Mention Energy'].mean()

In [ ]:
import seaborn as sns # another visualisations / graphing library

plt.figure(figsize = (14, 9))

sns.barplot(data = df2, x = 'Party', y = 'Mention Energy', palette='rocket', hue='Party', legend=False) # more colour palettes: https://seaborn.pydata.org/tutorial/color_palettes.html


plt.title('Average mentions of Energy per party', fontsize = 20)
plt.xlabel('Party', fontsize = 15)
plt.ylabel('Average number of mentions', fontsize = 15)

plt.show()

In [ ]:
# Let's get the mean number of mentions per party
df2.groupby('Party')['Mention Renewable'].mean()

In [ ]:
plt.figure(figsize = (14, 9))

sns.barplot(data = df2, x = 'Party', y = 'Mention Renewable', palette='tab10',  hue='Party', legend=False) # more colour palettes: https://seaborn.pydata.org/tutorial/color_palettes.html

plt.title('Average mentions of Renewable per party', fontsize = 20)
plt.xlabel('Party', fontsize = 15)
plt.ylabel('Average number of mentions', fontsize = 15)

plt.show()